**Save datasets to drive for easier usage**

In [ ]:
%%bash
mkdir -p /content/sinhala_audio
wget -q https://openslr.trmal.net/resources/52/utt_spk_text.tsv -O /content/utt_spk_text.tsv
for i in 0 1 2 3 4 5; do
  wget -q https://openslr.trmal.net/resources/52/asr_sinhala_${i}.zip -O /content/sinhala_${i}.zip
  unzip -q /content/sinhala_${i}.zip -d /content/sinhala_audio
  rm /content/sinhala_${i}.zip
done

In [4]:
%%bash
wget -q https://openslr.trmal.net/resources/12/train-clean-100.tar.gz -O /content/train-clean-100.tar.gz
tar -xzf /content/train-clean-100.tar.gz -C /content/
rm /content/train-clean-100.tar.gz

# Repackage as single tar.gz files for fast Drive storage/retrieval
tar -czf /content/sinhala_audio.tar.gz -C /content sinhala_audio
tar -czf /content/librispeech_clean100.tar.gz -C /content LibriSpeech

In [4]:
from google.colab import drive
drive.mount('/content/drive')

import os
DATA_ROOT = "/content/drive/MyDrive/Data_Science_Project/data"

Mounted at /content/drive


In [ ]:
# Upload zips to drive

import shutil
DATA_ROOT = "/content/drive/MyDrive/Data_Science_Project/data"

shutil.copy("/content/sinhala_audio.tar.gz", f"{DATA_ROOT}/sinhala_audio.tar.gz")
shutil.copy("/content/utt_spk_text.tsv", f"{DATA_ROOT}/utt_spk_text.tsv")
shutil.copy("/content/librispeech_clean100.tar.gz", f"{DATA_ROOT}/librispeech_clean100.tar.gz")

'/content/drive/MyDrive/Data_Science_Project/data/librispeech_clean100.tar.gz'

In [ ]:
!pip install -q speechbrain torch torchaudio pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.2/788.2 kB 59.2 MB/s eta 0:00:00


In [6]:
import shutil, subprocess
DATA_ROOT = "/content/drive/MyDrive/Data_Science_Project/data"

#shutil.copy(f"{DATA_ROOT}/sinhala_audio.tar.gz", "/content/sinhala_audio.tar.gz")
#shutil.copy(f"{DATA_ROOT}/utt_spk_text.tsv", "/content/utt_spk_text.tsv")
#shutil.copy(f"{DATA_ROOT}/librispeech_clean100.tar.gz", "/content/librispeech_clean100.tar.gz")

subprocess.run(["tar", "-xzf", "/content/sinhala_audio.tar.gz", "-C", "/content/"])
subprocess.run(["tar", "-xzf", "/content/librispeech_clean100.tar.gz", "-C", "/content/"])

CompletedProcess(args=['tar', '-xzf', '/content/librispeech_clean100.tar.gz', '-C', '/content/'], returncode=0)

**PHASE 2 -- Baseline bilingual speaker verification (Sinhala + English)**

1. Discovers whatever Sinhala (SLR52) and English (LibriSpeech train-clean-100)
   audio you've downloaded locally, and reports how many speakers/utterances
   are actually usable -- so you know if you need more shards.
2. Builds balanced genuine/impostor verification trials separately per language,
   using ALL available speakers (diversity-first, matching earlier guidance).
3. Runs both ECAPA-TDNN and X-Vector (SpeechBrain pretrained models).
4. Reports EER, FAR/FRR-relevant similarity stats, per model per language.

In [ ]:
import glob
import itertools
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torchaudio

try:
    from speechbrain.inference.speaker import EncoderClassifier
except ImportError:
    from speechbrain.pretrained import EncoderClassifier

SAMPLE_RATE = 16000

In [ ]:
# =====================================================================
# DATA DISCOVERY
# =====================================================================

def discover_sinhala_speakers(audio_dir: str, tsv_path: str) -> dict:
    """
    Returns {speaker_id: [wav_path, ...]} using only files that are
    actually present on disk (matches FileID from the tsv to files found
    by recursive glob, regardless of exact folder structure the zip
    extracted into).
    """
    metadata = pd.read_csv(tsv_path, sep="\t", header=None,
                            names=["FileID", "UserID", "Transcription"])

    found_files = glob.glob(str(Path(audio_dir) / "**" / "*"), recursive=True)
    found_files = [f for f in found_files if f.lower().endswith((".wav", ".flac"))]
    stem_to_path = {Path(f).stem: f for f in found_files}

    metadata["path"] = metadata["FileID"].map(stem_to_path)
    available = metadata.dropna(subset=["path"])

    speaker_to_files = available.groupby("UserID")["path"].apply(list).to_dict()
    print(f"[Sinhala] {len(found_files)} audio files found on disk")
    print(f"[Sinhala] {len(speaker_to_files)} speakers have at least 1 locally available file")
    return speaker_to_files


def discover_librispeech_speakers(root_dir: str) -> dict:
    """
    LibriSpeech layout: <root>/<speaker_id>/<chapter_id>/<speaker>-<chapter>-<utt>.flac
    Returns {speaker_id: [wav_path, ...]}
    """
    speaker_to_files = {}
    for speaker_dir in sorted(Path(root_dir).iterdir()):
        if not speaker_dir.is_dir():
            continue
        files = glob.glob(str(speaker_dir / "**" / "*.flac"), recursive=True)
        if files:
            speaker_to_files[speaker_dir.name] = files
    print(f"[English] {len(speaker_to_files)} speakers found under {root_dir}")
    return speaker_to_files


def report_sufficiency(name: str, speaker_to_files: dict, target_speakers: int = 100,
                        target_utts: int = 4):
    n_speakers = len(speaker_to_files)
    n_eligible = sum(1 for files in speaker_to_files.values() if len(files) >= 2)
    n_full = sum(1 for files in speaker_to_files.values() if len(files) >= target_utts)
    print(f"[{name}] {n_speakers} total speakers | {n_eligible} with >=2 utterances "
          f"| {n_full} with >={target_utts} utterances")
    if n_full < target_speakers:
        print(f"[{name}] WARNING: only {n_full} speakers meet the target of "
              f"{target_utts}+ utterances (recommended: {target_speakers}+ speakers). "
              f"Consider downloading more shards/data.")
    else:
        print(f"[{name}] OK -- sufficient speaker diversity for a stable EER estimate.")


In [ ]:
# =====================================================================
# TRIAL GENERATION (speaker-diversity-first, as established earlier)
# =====================================================================
def build_balanced_trials(speaker_to_files: dict, utts_per_speaker: int, seed: int = 42):
    rng = random.Random(seed)
    eligible = {s: files for s, files in speaker_to_files.items() if len(files) >= 2}
    capped = {s: files[:utts_per_speaker] for s, files in eligible.items()}

    positive_pairs = []
    for spk, files in capped.items():
        positive_pairs.extend(itertools.combinations(files, 2))
    rng.shuffle(positive_pairs)

    speakers = list(capped.keys())
    file_to_speaker = {f: s for s, files in capped.items() for f in files}

    negative_pairs = []
    for f1, _ in positive_pairs:
        spk1 = file_to_speaker[f1]
        other_spk = rng.choice([s for s in speakers if s != spk1])
        other_file = rng.choice(capped[other_spk])
        negative_pairs.append((f1, other_file))

    eval_files = list({f for pair in positive_pairs for f in pair} |
                       {f for pair in negative_pairs for f in pair})

    print(f"   -> {len(capped)} speakers used, {len(eval_files)} unique files, "
          f"{len(positive_pairs)} positive / {len(negative_pairs)} negative pairs")
    return positive_pairs, negative_pairs, eval_files


In [ ]:
# =====================================================================
# AUDIO LOADING + EMBEDDING + METRICS
# =====================================================================
def load_mono_16k(path: str) -> np.ndarray:
    waveform, sr = torchaudio.load(path)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    if sr != SAMPLE_RATE:
        waveform = torchaudio.functional.resample(waveform, sr, SAMPLE_RATE)
    return waveform.squeeze(0).numpy()


def get_embedding(audio_array: np.ndarray, classifier, device) -> torch.Tensor:
    signal = torch.tensor(audio_array).unsqueeze(0).float().to(device)
    with torch.no_grad():
        emb = classifier.encode_batch(signal).squeeze(1)
        return F.normalize(emb, p=2, dim=-1)


def calculate_eer(pos_scores, neg_scores):
    if len(pos_scores) == 0 or len(neg_scores) == 0:
        return 0.0, 0.0
    pos, neg = np.array(pos_scores), np.array(neg_scores)
    thresholds = np.sort(np.concatenate([pos, neg]))
    min_diff, best_eer, best_thresh = float("inf"), 1.0, 0.0
    for t in thresholds:
        far = np.mean(neg >= t)
        frr = np.mean(pos < t)
        diff = abs(far - frr)
        if diff < min_diff:
            min_diff, best_eer, best_thresh = diff, (far + frr) / 2.0, t
    return best_eer * 100, best_thresh


def evaluate_language(name, positive_pairs, negative_pairs, eval_files, models: dict, device):
    print(f"\n--- Extracting embeddings: {name} ---")
    caches = {model_name: {} for model_name in models}

    for idx, path in enumerate(eval_files):
        if idx % 100 == 0 or idx == len(eval_files) - 1:
            print(f"   -> {idx + 1}/{len(eval_files)}")
        try:
            audio = load_mono_16k(path)
        except Exception as e:
            print(f"      [Warning] Could not load {path}: {e}")
            continue
        embeddings = {}
        ok = True
        for model_name, model in models.items():
            try:
                embeddings[model_name] = get_embedding(audio, model, device)
            except Exception as e:
                print(f"      [Warning] Embedding failed for {path} ({model_name}): {e}")
                ok = False
                break
        if ok:
            for model_name in models:
                caches[model_name][path] = embeddings[model_name]

    cos = torch.nn.CosineSimilarity(dim=-1)
    results = {}
    for model_name, cache in caches.items():
        pos_scores = [cos(cache[a], cache[b]).item()
                      for a, b in positive_pairs if a in cache and b in cache]
        neg_scores = [cos(cache[a], cache[b]).item()
                      for a, b in negative_pairs if a in cache and b in cache]
        avg_pos = float(np.mean(pos_scores)) if pos_scores else 0.0
        avg_neg = float(np.mean(neg_scores)) if neg_scores else 0.0
        eer, thresh = calculate_eer(pos_scores, neg_scores)
        results[model_name] = {
            "avg_pos": avg_pos, "avg_neg": avg_neg, "margin": avg_pos - avg_neg,
            "eer": eer, "thresh": thresh, "n_pos": len(pos_scores), "n_neg": len(neg_scores),
        }
    return results


In [ ]:
# =====================================================================
# MAIN
# =====================================================================
def main(args):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device.type.upper()}")

    print("\n=== Loading pretrained models ===")
    
    ecapa = EncoderClassifier.from_hparams(
        source="/content/drive/MyDrive/Data_Science_Project/Models/ECAPA-TDNN",
        savedir="tmpdir_ecapa", run_opts={"device": str(device)},
    )

    models = {"ECAPA-TDNN": ecapa}

    print("\n=== Discovering data ===")
    sinhala_speakers = discover_sinhala_speakers(args["sinhala_audio_dir"], args["sinhala_tsv"])
    english_speakers = discover_librispeech_speakers(args["english_dir"])
    report_sufficiency("Sinhala", sinhala_speakers, target_utts=args["utts_per_speaker"])
    report_sufficiency("English", english_speakers, target_utts=args["utts_per_speaker"])

    print("\n=== Building trials ===")
    print("Sinhala:")
    sin_pos, sin_neg, sin_eval = build_balanced_trials(sinhala_speakers, args["utts_per_speaker"], args["seed"])
    print("English:")
    eng_pos, eng_neg, eng_eval = build_balanced_trials(english_speakers, args["utts_per_speaker"], args["seed"])

    sinhala_results = evaluate_language("Sinhala", sin_pos, sin_neg, sin_eval, models, device)
    english_results = evaluate_language("English", eng_pos, eng_neg, eng_eval, models, device)

    print("\n" + "=" * 90)
    print("PHASE 2 BASELINE RESULTS")
    print("=" * 90)
    rows = ["Average Same-Speaker Sim", "Average Diff-Speaker Sim",
            "Separation Margin", "Decision Threshold", "EER (%)", "N pos / N neg"]

    def col(res, model):
        r = res[model]
        return [f"{r['avg_pos']:.4f}", f"{r['avg_neg']:.4f}", f"{r['margin']:.4f}",
                f"{r['thresh']:.4f}", f"{r['eer']:.2f}%", f"{r['n_pos']} / {r['n_neg']}"]

    df = pd.DataFrame({
        "Metric": rows,
        "Sinhala: ECAPA": col(sinhala_results, "ECAPA-TDNN"),
        "English: ECAPA": col(english_results, "ECAPA-TDNN"),
    })
    print(df.to_string(index=False))
    print("=" * 90)

    df.to_csv("phase2_baseline_results.csv", index=False)
    print("\nSaved: phase2_baseline_results.csv")
    return df

In [13]:
if __name__ == "__main__":
    args = {
        "sinhala_audio_dir": "./sinhala_audio",
        "sinhala_tsv": "./utt_spk_text.tsv",
        "english_dir": "./LibriSpeech/train-clean-100",
        "utts_per_speaker": 5,
        "seed": 42
    }
    main(args)

INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Using symlink found at '/content/tmpdir_ecapa/hyperparams.yaml'
INFO:speechbrain.utils.fetching:Fetch embedding_model.ckpt: Using symlink found at '/content/tmpdir_ecapa/embedding_model.ckpt'
INFO:speechbrain.utils.fetching:Fetch mean_var_norm_emb.ckpt: Using symlink found at '/content/tmpdir_ecapa/mean_var_norm_emb.ckpt'
INFO:speechbrain.utils.fetching:Fetch classifier.ckpt: Using symlink found at '/content/tmpdir_ecapa/classifier.ckpt'
INFO:speechbrain.utils.fetching:Fetch label_encoder.txt: Using symlink found at '/content/tmpdir_ecapa/label_encoder.ckpt'
INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: embedding_model, mean_var_norm_emb, classifier, label_encoder


Device: CUDA

=== Loading pretrained models ===



=== Discovering data ===
[Sinhala] 69268 audio files found on disk
[Sinhala] 478 speakers have at least 1 locally available file
[English] 251 speakers found under ./LibriSpeech/train-clean-100
[Sinhala] 478 total speakers | 478 with >=2 utterances | 477 with >=5 utterances
[Sinhala] OK -- sufficient speaker diversity for a stable EER estimate.
[English] 251 total speakers | 251 with >=2 utterances | 251 with >=5 utterances
[English] OK -- sufficient speaker diversity for a stable EER estimate.

=== Building trials ===
Sinhala:
   -> 478 speakers used, 2389 unique files, 4776 positive / 4776 negative pairs
English:
   -> 251 speakers used, 1255 unique files, 2510 positive / 2510 negative pairs

--- Extracting embeddings: Sinhala ---
   -> 1/2389
   -> 101/2389
   -> 201/2389
   -> 301/2389
   -> 401/2389
   -> 501/2389
   -> 601/2389
   -> 701/2389
   -> 801/2389
   -> 901/2389
   -> 1001/2389
   -> 1101/2389
   -> 1201/2389
   -> 1301/2389
   -> 1401/2389
   -> 1501/2389
   -> 1601/2

## Phase 2 Baseline Results

| Metric | Sinhala: ECAPA | English: ECAPA |
| :--- | :--- | :--- |
| **Average Same-Speaker Sim** | 0.5857 | 0.8296 |
| **Average Diff-Speaker Sim** | 0.1319 | 0.0969 |
| **Separation Margin** | 0.4537 | 0.7328 |
| **Decision Threshold** | 0.3519 | 0.4328 |
| **EER (%)** | 4.52% | 0.36% |
| **N pos / N neg** | 4776 / 4776 | 2510 / 2510 |

In [14]:
from google.colab import runtime
runtime.unassign()